# 05 — Pelabelan Kandidat Kelas Baru: Oracle + LLM Auto-Name (Paper 3, Tahap T4)

**Dijalankan di SageMaker.** Menjawab: *cluster kandidat (hasil nb 03) diberi label apa?*
Dua jalur, dgn pembagian peran yg menjaga klaim inti tetap kokoh (documentation.md §6):

1. **Oracle terjadwal (JALUR UTAMA).** Ground-truth per flow (`attack_cat` held-out) sudah
   diketahui → label cluster = kelas **mayoritas** anggotanya. 100% dasar pembuktian;
   **tidak bergantung** LLM. Ini yang dipakai untuk update model (nb 06).
2. **LLM auto-name (KONTRIBUSI TAMBAHAN).** Dari **profil statistik cluster**
   (`cluster_profile_<DS>.csv` nb 03), LLM memetakan profil → taksonomi serangan
   (mis. 'slow-rate DoS'). LLM = *penamaan cluster*, BUKAN classifier per-flow.
   **Akurasi LLM diukur vs oracle** (mis. benar 2 dari 3 cluster).

**Robustness:** bila API LLM tak tersedia (tanpa key/koneksi), otomatis pakai
**rule-based namer** (heuristik profil) sbg pengganti, tetap dibandingkan ke oracle.
Metode & angka dilaporkan apa adanya (documentation.md: LLM dari 9 angka statistik terbatas).

**Output** (→ S3 `evolusion/labeling/`): `labeling_results.json`, `labeling_<DS>.csv`
(cluster → oracle_label, llm_name, match?).

> Semua angka dari eksekusi nyata. Kandidat cluster + ground-truth berasal dari nb 03.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('pandas','numpy','boto3') if u.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='evolusion'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
IN03='novelty_out'; OUTDIR='labeling_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
# Konfigurasi LLM (opsional). Jika USE_LLM=False atau API gagal -> fallback rule-based.
USE_LLM=os.environ.get('USE_LLM','0')=='1'          # set '1' utk aktifkan LLM (Bedrock)
LLM_MODEL=os.environ.get('LLM_MODEL','anthropic.claude-3-5-sonnet-20240620-v1:0')
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','use_llm_requested':USE_LLM}
def fetch03(fname):
    lp=os.path.join(IN03,fname)
    if os.path.exists(lp): return lp
    try:
        import boto3; os.makedirs(IN03,exist_ok=True)
        boto3.client('s3',region_name=REGION).download_file(S3_BUCKET,f'{S3_PREFIX}/novelty/{fname}',lp)
        print('   diunduh dari S3:',fname); return lp
    except Exception as e: print('   GAGAL ambil',fname,'->',e); return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Oracle: label cluster = kelas mayoritas (dari komposisi ground-truth nb 03)

In [ ]:
def load_novelty():
    p=fetch03('novelty_results.json')
    if not p: print('novelty_results.json tak ada; jalankan nb 03 dulu'); return None
    return json.load(open(p))

def oracle_labels(ds_res):
    """Label oracle tiap cluster = kelas ground-truth MAYORITAS (dari cluster_composition nb 03)."""
    out={}
    for cid,comp in ds_res.get('cluster_composition',{}).items():
        if not comp: continue
        maj=max(comp.items(), key=lambda kv: kv[1])
        total=sum(comp.values())
        out[int(cid)]={'oracle_label':maj[0],'purity':round(maj[1]/total,4),'n':total,'composition':comp}
    return out

nov=load_novelty()
print('dataset tersedia di novelty:', list((nov or {}).get('novelty',{}).keys()) if nov else None)
print('=== SEL 2 (oracle) SELESAI ===')

## 3. Namer: LLM (Bedrock) atau fallback rule-based dari profil statistik

In [ ]:
def describe_profile(row):
    """Ubah profil median 9-fitur -> deskripsi teks singkat (untuk prompt LLM & rule-based)."""
    d=row.to_dict()
    return (f"durasi_us={d.get('duration',0):.0f}, fwd_pkts={d.get('fwd_pkts',0):.1f}, "
            f"bwd_pkts={d.get('bwd_pkts',0):.1f}, fwd_bytes={d.get('fwd_bytes',0):.0f}, "
            f"bwd_bytes={d.get('bwd_bytes',0):.0f}, fwd_mean={d.get('fwd_mean',0):.1f}, "
            f"bwd_mean={d.get('bwd_mean',0):.1f}, src_load={d.get('src_load',0):.1f}, "
            f"dst_load={d.get('dst_load',0):.1f}")

def rule_based_name(row):
    """Heuristik sederhana profil -> nama kasar. JUJUR: ini fallback lemah (dari 9 angka)."""
    d=row.to_dict()
    dur=d.get('duration',0); fp=d.get('fwd_pkts',0); bp=d.get('bwd_pkts',0)
    sl=d.get('src_load',0); fb=d.get('fwd_bytes',0)
    if dur>5e7 and fp<10: return 'slow-rate/low-volume (mis. Slowloris-like)'
    if sl>1e6 or (fp+bp)>500: return 'high-rate/flood (mis. DoS/DDoS-like)'
    if fb<100 and fp<5: return 'probe/scan-like (paket kecil, sedikit)'
    return 'unknown-pattern (profil tak khas)'

def llm_name(desc_list):
    """Panggil Bedrock; kembalikan list nama utk tiap cluster. Gagal -> None (caller fallback)."""
    try:
        import boto3
        br=boto3.client('bedrock-runtime',region_name=REGION)
        prompt=("Anda analis keamanan jaringan. Diberi profil statistik cluster flow (fitur agregat), "
                "petakan tiap cluster ke SATU jenis serangan jaringan yang paling mungkin "
                "(mis. Slowloris, Hulk, SYN-flood, UDP-flood, port-scan, brute-force, botnet, dll). "
                "Jawab JSON list of string, urut sesuai input, tanpa penjelasan.\n\nProfil cluster:\n"+
                "\n".join(f"{i}: {d}" for i,d in enumerate(desc_list)))
        body=json.dumps({'anthropic_version':'bedrock-2023-05-31','max_tokens':300,
                         'messages':[{'role':'user','content':prompt}]})
        r=br.invoke_model(modelId=LLM_MODEL, body=body)
        txt=json.loads(r['body'].read())['content'][0]['text']
        s=txt[txt.find('['):txt.rfind(']')+1]
        names=json.loads(s)
        return [str(x) for x in names]
    except Exception as e:
        print('   LLM gagal/nonaktif -> fallback rule-based:',str(e)[:120]); return None
print('=== SEL 3 (namer) SELESAI ===')

## 4. Jalankan pelabelan per dataset + ukur akurasi LLM vs oracle

In [ ]:
def norm(s): return str(s).strip().lower()

def name_matches_oracle(llm, oracle):
    """Cocok longgar: kata kunci oracle muncul di nama LLM (taksonomi beda kata).
    JUJUR: ini penilaian longgar; disebut eksplisit di paper."""
    o=norm(oracle); l=norm(llm)
    keys={'dos':['dos','slow','hulk','flood','goldeneye'],'ddos':['ddos','flood','syn','udp'],
          'botnet':['bot'],'infiltration':['infil','backdoor','exploit'],
          'worms':['worm'],'shellcode':['shell','exploit'],'backdoor':['backdoor','exploit'],
          'bruteforce':['brute'],'recon':['scan','probe','recon'],'exploits':['exploit'],
          'fuzzers':['fuzz'],'generic':['generic']}
    for k,alts in keys.items():
        if k in o: return any(a in l for a in alts)
    return o in l

def run_labeling(ds, ds_res):
    prof_path=fetch03(f'cluster_profile_{ds}.csv')
    if not prof_path: print(f'[{ds}] cluster_profile tak ada'); return None
    prof=pd.read_csv(prof_path).set_index('cluster')
    orc=oracle_labels(ds_res)
    cids=[c for c in prof.index if c in orc]
    if not cids: print(f'[{ds}] tak ada cluster valid'); return None
    descs=[describe_profile(prof.loc[c]) for c in cids]
    names=llm_name(descs) if USE_LLM else None
    used_llm = names is not None and len(names)==len(cids)
    if not used_llm:
        names=[rule_based_name(prof.loc[c]) for c in cids]
    rows=[]; n_match=0
    for c,nm in zip(cids,names):
        ol=orc[c]['oracle_label']; m=name_matches_oracle(nm,ol); n_match+=int(m)
        rows.append({'cluster':c,'n':orc[c]['n'],'oracle_label':ol,'purity':orc[c]['purity'],
                     'name':nm,'match_oracle':m})
    df=pd.DataFrame(rows)
    df.to_csv(os.path.join(OUTDIR,f'labeling_{ds}.csv'),index=False)
    acc=round(n_match/len(cids),4)
    res={'dataset':ds,'namer':'LLM' if used_llm else 'rule-based','n_clusters':len(cids),
         'n_match_oracle':n_match,'name_accuracy_vs_oracle':acc,
         'clusters':rows}
    print(f"[{ds}] namer={res['namer']} | cocok {n_match}/{len(cids)} cluster (akurasi={acc}) vs oracle")
    import IPython.display as ipd; ipd.display(df)
    return res

RESULTS['labeling']={}
if nov:
    for ds,ds_res in nov.get('novelty',{}).items():
        r=run_labeling(ds,ds_res)
        if r: RESULTS['labeling'][ds]=r
print('=== SEL 4 (pelabelan) SELESAI ===')

## 5. Simpan + UPLOAD S3

In [ ]:
jp=os.path.join(OUTDIR,'labeling_results.json'); json.dump(RESULTS,open(jp,'w'),indent=2); print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/labeling/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/labeling/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 5 (simpan + upload) SELESAI ===')
print('SELESAI T4. Label ORACLE (mayoritas) dipakai utk update model (nb 06). '
      'Akurasi LLM vs oracle = kontribusi tambahan (dilaporkan apa adanya).')